# Per-Bone Ground-Truth Build & Alignment

Voxelize manual Slicer STLs (femur/tibia/patella/fibula) onto each case's predrr grid,
write per-bone `.nii.gz` for Slicer inspection, and validate alignment before modelling.

In [1]:
from pathlib import Path
import re, numpy as np, nibabel as nib, trimesh
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

BONES = ["femur", "tibia", "patella", "fibula"]
STL_ROOT  = ROOT / "data/external/ground_truth/fracture_ground_truth"
PREDRR    = ROOT / "data/interim/predrr/fractured"
OUT_ROOT  = ROOT / "data/interim/gt_per_bone_256/fractured"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# folder-stem (Case<N>) -> predrr KEY (side comes from predrr, not the STL token)
CASE_TO_KEY = {p.name.split("_Part")[0]: p.stem.replace(".nii", "")
               for p in PREDRR.glob("*.nii.gz")}
print("predrr keys:", sorted(CASE_TO_KEY.values()))

predrr keys: ['Case11_PartRight', 'Case12_PartRight', 'Case13_PartRight', 'Case14_PartRight', 'Case15_PartRight', 'Case16_PartRight', 'Case1_PartLeft', 'Case2_PartLeft', 'Case3_PartLeft', 'Case5_PartRight', 'Case6_PartRight', 'Case7_PartRight', 'Case9_PartRight']


In [2]:
def parse_bone(fname: str) -> str:
    """Map a messy STL filename to one of BONES by substring, ignoring side/segmentation noise."""
    s = fname.lower()
    for bone in BONES:
        if bone in s:                  # 'femur','tibia','patella','fibula'
            return bone
    raise ValueError(f"no bone token in {fname!r}")

In [3]:
def _case_stem(folder_name: str) -> str:
    # "Case2 (does not seem fractured)" -> "Case2" ; "Case11" -> "Case11"
    return re.match(r"(Case\d+)", folder_name).group(1)

cases = {}
for folder in sorted(STL_ROOT.iterdir()):
    if not folder.is_dir():
        continue
    stem = _case_stem(folder.name)
    if stem not in CASE_TO_KEY:        # e.g. no predrr -> skip
        print("SKIP (no predrr):", folder.name); continue
    bone_map = {}
    for stl in folder.glob("*.stl"):
        bone = parse_bone(stl.name)
        assert bone in BONES, f"bad bone {bone} for {stl.name}"
        assert bone not in bone_map, f"duplicate {bone} in {folder.name}: {stl.name} vs {bone_map[bone].name}"
        bone_map[bone] = stl
    missing = set(BONES) - set(bone_map)
    print(f"{folder.name:45s} -> {CASE_TO_KEY[stem]:18s} bones={sorted(bone_map)} missing={sorted(missing)}")
    cases[CASE_TO_KEY[stem]] = bone_map
assert cases, "no cases mapped"

Case1                                         -> Case1_PartLeft     bones=['femur', 'fibula', 'patella', 'tibia'] missing=[]
Case11                                        -> Case11_PartRight   bones=['femur', 'fibula', 'patella', 'tibia'] missing=[]
Case12                                        -> Case12_PartRight   bones=['femur', 'fibula', 'patella', 'tibia'] missing=[]
Case13                                        -> Case13_PartRight   bones=['femur', 'fibula', 'patella', 'tibia'] missing=[]
Case16                                        -> Case16_PartRight   bones=['femur', 'fibula', 'patella', 'tibia'] missing=[]
Case2 (does not seem fractured)               -> Case2_PartLeft     bones=['femur', 'fibula', 'patella', 'tibia'] missing=[]
Case3                                         -> Case3_PartLeft     bones=['femur', 'fibula', 'patella', 'tibia'] missing=[]
Case5                                         -> Case5_PartRight    bones=['femur', 'fibula', 'patella', 'tibia'] missing=[]


In [ ]:
def voxelize_bone(stl_path: Path, predrr_img) -> np.ndarray:
    """STL (world RAS mm) -> uint8 occupancy on predrr voxel grid via inv(affine)."""
    inv = np.linalg.inv(predrr_img.affine)
    mesh = trimesh.load(stl_path, process=False)
    v = np.asarray(mesh.vertices)
    vh = np.c_[v, np.ones(len(v))]
    mesh.vertices = (inv @ vh.T).T[:, :3]          # now in voxel-index space
    vg = mesh.voxelized(pitch=1.0).fill()          # solid fill in index space
    idx = np.round(np.asarray(vg.points)).astype(int)
    shape = predrr_img.shape
    mask = np.zeros(shape, dtype=np.uint8)
    inb = (idx >= 0).all(1) & (idx[:, 0] < shape[0]) & (idx[:, 1] < shape[1]) & (idx[:, 2] < shape[2])
    idx = idx[inb]
    mask[idx[:, 0], idx[:, 1], idx[:, 2]] = 1
    return mask

In [ ]:
key  = sorted(cases)[0]
img  = nib.load(PREDRR / f"{key}.nii.gz")
ct   = np.asarray(img.dataobj)
mask = voxelize_bone(cases[key]["femur"], img)
print(key, "femur voxels:", int(mask.sum()), "ct shape:", ct.shape)
assert mask.sum() > 1000, "femur mask implausibly small — voxelization/fill failed"

# overlay at the slice with the most femur signal, per axis
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, axis in enumerate([0, 1, 2]):
    s = mask.sum(axis=tuple(i for i in range(3) if i != axis)).argmax()
    ct_sl   = np.take(ct,   s, axis=axis)
    mask_sl = np.take(mask, s, axis=axis)
    ax[a].imshow(ct_sl.T, cmap="gray", origin="lower")
    ax[a].imshow(np.ma.masked_where(mask_sl.T == 0, mask_sl.T), cmap="autumn", alpha=0.5, origin="lower")
    ax[a].set_title(f"{key} femur axis{axis} slice{s}")
plt.tight_layout()
plt.savefig(OUT_ROOT / f"_spotcheck_{key}_femur.png", dpi=110)   # <-- save for review
plt.show()